# L87 · 量化与本地推理 — INT8 量化原理，连接 HuggingFace 轻量推理

**学习目标**
- 理解 float32 → int8 量化的原理（scale + zero_point）
- 从零实现 INT8 量化/反量化
- 测量量化误差 vs 比特数
- （可选）用 HuggingFace 连接真实本地 LLM

## 为什么量化？

7B 参数模型 × 4 bytes（float32）= 28 GB VRAM → 需要 A100。
7B 参数模型 × 1 byte（int8）   =  7 GB VRAM → 可以放在 RTX 3080。

**量化公式**：
```
scale     = (x_max - x_min) / (2^bits - 1)
zero_point = round(-x_min / scale)  （使 0 精确可表示）
x_q       = round(x / scale) + zero_point
x_deq     = (x_q - zero_point) × scale   （反量化）
```

量化误差（量化噪声）≈ scale/2，越少比特误差越大。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def quantize_int8(x: np.ndarray):
    """将 float32 权重量化为 int8，返回 (quantized, scale, zero_point)。"""
    x_min, x_max = x.min(), x.max()
    scale = (x_max - x_min) / 255.0  # int8 范围 0-255 (unsigned)
    if scale < 1e-8:
        return np.zeros_like(x, dtype=np.int8), 1.0, 0
    zero_point = int(round(-x_min / scale))
    x_q = np.round(x / scale).astype(np.int32) + zero_point
    x_q = np.clip(x_q, 0, 255).astype(np.uint8)
    return x_q, scale, zero_point

def dequantize_int8(x_q: np.ndarray, scale: float, zero_point: int) -> np.ndarray:
    """反量化：int8 → float32。"""
    return (x_q.astype(np.float32) - zero_point) * scale

# 测试
np.random.seed(0)
W = np.random.randn(64, 64).astype(np.float32)   # 模拟权重矩阵

W_q, scale, zp = quantize_int8(W)
W_deq = dequantize_int8(W_q, scale, zp)
error = np.abs(W - W_deq)

print(f'原始范围: [{W.min():.3f}, {W.max():.3f}]')
print(f'Scale: {scale:.6f}, Zero point: {zp}')
print(f'量化误差: mean={error.mean():.4f}, max={error.max():.4f}')
print(f'内存: float32={W.nbytes} bytes → int8={W_q.nbytes} bytes  ({W.nbytes/W_q.nbytes:.1f}× 压缩)')

In [ ]:
# 误差 vs 比特数
print(f'{"位宽":>4}  {"levels":>7}  {"均值误差":>10}  {"最大误差":>10}')
print('-' * 40)
for bits in [8, 6, 4, 3, 2]:
    levels = 2**bits - 1
    scale_b = (W.max() - W.min()) / levels
    W_q_b = np.clip(np.round((W - W.min()) / scale_b), 0, levels)
    W_r = W_q_b * scale_b + W.min()
    err = np.abs(W - W_r)
    print(f'{bits:>4}  {levels:>7}  {err.mean():>10.4f}  {err.max():>10.4f}')

## 可选：连接 HuggingFace 本地推理

> 以下代码需要安装 `transformers` 和适当的模型权重。
> 如果跳过，前面的量化原理部分已经完整展示核心算法。

⚠️ **注意**：`model.generate()` 是 HuggingFace 的封装，这里作为工程集成使用，核心推理算法（KV-Cache、采样）见 L85-L86。

In [ ]:
# 工程集成示例（需要 transformers）
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch

    # 使用小型模型演示（替换为你有的本地模型）
    MODEL_ID = 'Qwen/Qwen2.5-0.5B'   # ~1GB，可本地运行

    print(f'Loading {MODEL_ID}...')
    # 实际运行时取消注释：
    # tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    # model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    # inputs = tokenizer('The Fourier transform', return_tensors='pt')
    # with torch.no_grad():
    #     out = model.generate(**inputs, max_new_tokens=30, temperature=0.8, do_sample=True)
    # print(tokenizer.decode(out[0], skip_special_tokens=True))
    print('（已注释 — 取消注释并有对应模型权重后可运行）')
    print('核心算法（KV-Cache、采样）见 L85 和 L86 的从零实现')

except ImportError:
    print('transformers 未安装 — 跳过工程集成部分')
    print('前面的量化原理（INT8 quantize/dequantize）已完整运行')

## 小结

| 概念 | 要记住的 |
|---|---|
| INT8 量化 | scale = range/255, error ≈ scale/2 |
| 4× 内存压缩 | float32 → int8：显存需求降到 1/4 |
| 权衡 | 更少比特 → 更小模型 → 更大量化误差 |
| 工程集成 | HuggingFace = 便捷封装，不替代从零理解 |
| L88 | RAG 检索——不需要大模型也能做智能问答 |